# Model Comparison and Inference Demo

This notebook demonstrates:
- Loading trained models
- Comparing model architectures and performance
- Running inference on sample images
- Visualizing model predictions

## 1. Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.datasets import mnist
import tensorflow_datasets as tfds

plt.style.use('dark_background')
sns.set_palette('husl')

CHECKPOINT_DIR = '../backend/checkpoints'
CHAR_LABELS = [chr(ord('A') + i) for i in range(26)]

## 2. Load Models

In [ ]:
# Load digit models
digit_model_path = os.path.join(CHECKPOINT_DIR, 'cnn_best.keras')
baseline_model_path = os.path.join(CHECKPOINT_DIR, 'baseline_best.keras')

if os.path.exists(digit_model_path):
    digit_cnn = tf.keras.models.load_model(digit_model_path)
    print(f'✓ Digit CNN loaded: {digit_model_path}')
else:
    digit_cnn = None
    print('✗ Digit CNN not found - run training first')

if os.path.exists(baseline_model_path):
    baseline_model = tf.keras.models.load_model(baseline_model_path)
    print(f'✓ Baseline ANN loaded: {baseline_model_path}')
else:
    baseline_model = None
    print('✗ Baseline ANN not found')

In [ ]:
# Load character model
char_model_path = os.path.join(CHECKPOINT_DIR, 'char_cnn_best.keras')

if os.path.exists(char_model_path):
    char_cnn = tf.keras.models.load_model(char_model_path)
    print(f'✓ Character CNN loaded: {char_model_path}')
else:
    char_cnn = None
    print('✗ Character CNN not found - run character training first')

## 3. Model Architecture Comparison

In [ ]:
def model_stats(model, name):
    if model is None:
        return {'name': name, 'params': 0, 'layers': 0}
    return {
        'name': name,
        'params': model.count_params(),
        'layers': len(model.layers)
    }

stats = [
    model_stats(baseline_model, 'Baseline ANN'),
    model_stats(digit_cnn, 'Digit CNN'),
    model_stats(char_cnn, 'Character CNN')
]

print('=' * 50)
print('MODEL COMPARISON')
print('=' * 50)
print(f'{"Model":<20} {"Parameters":>15} {"Layers":>10}')
print('-' * 50)
for s in stats:
    print(f'{s["name"]:<20} {s["params"]:>15,} {s["layers"]:>10}')

In [ ]:
# Visualize parameter counts
names = [s['name'] for s in stats if s['params'] > 0]
params = [s['params'] for s in stats if s['params'] > 0]

if names:
    plt.figure(figsize=(10, 5))
    bars = plt.bar(names, params, color=['#3498db', '#e74c3c', '#2ecc71'][:len(names)])
    plt.ylabel('Parameters')
    plt.title('Model Size Comparison')
    
    for bar, p in zip(bars, params):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
                f'{p:,}', ha='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()

## 4. Load Test Data

In [ ]:
# MNIST test data
(_, _), (X_test_digits, y_test_digits) = mnist.load_data()
X_test_digits = X_test_digits.astype('float32') / 255.0
X_test_digits = X_test_digits.reshape(-1, 28, 28, 1)

print(f'MNIST test samples: {len(X_test_digits)}')

In [ ]:
# EMNIST test data (if needed)
try:
    _, ds_test = tfds.load('emnist/letters', split=['train', 'test'], as_supervised=True)
    
    X_test_chars = []
    y_test_chars = []
    for img, label in tfds.as_numpy(ds_test):
        X_test_chars.append(img)
        y_test_chars.append(label)
    
    X_test_chars = np.array(X_test_chars)
    y_test_chars = np.array(y_test_chars) - 1  # 1-26 -> 0-25
    
    # Transpose and normalize
    X_test_chars = np.transpose(X_test_chars[:, :, :, 0], (0, 2, 1))
    X_test_chars = X_test_chars.astype('float32') / 255.0
    X_test_chars = X_test_chars.reshape(-1, 28, 28, 1)
    
    print(f'EMNIST test samples: {len(X_test_chars)}')
except:
    X_test_chars = None
    y_test_chars = None
    print('EMNIST not loaded (download if needed)')

## 5. Digit Recognition Demo

In [ ]:
if digit_cnn is not None:
    # Random digit predictions
    fig, axes = plt.subplots(2, 5, figsize=(12, 6))
    fig.suptitle('Digit CNN Predictions', fontsize=14)
    
    indices = np.random.choice(len(X_test_digits), 10, replace=False)
    
    for i, ax in enumerate(axes.flat):
        idx = indices[i]
        img = X_test_digits[idx]
        
        pred = digit_cnn.predict(img.reshape(1, 28, 28, 1), verbose=0)
        pred_label = np.argmax(pred[0])
        conf = np.max(pred[0]) * 100
        
        ax.imshow(img.squeeze(), cmap='gray')
        color = 'lime' if pred_label == y_test_digits[idx] else 'red'
        ax.set_title(f'Pred: {pred_label} ({conf:.1f}%)', color=color)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print('Digit CNN not loaded')

## 6. Character Recognition Demo

In [ ]:
if char_cnn is not None and X_test_chars is not None:
    fig, axes = plt.subplots(2, 5, figsize=(12, 6))
    fig.suptitle('Character CNN Predictions', fontsize=14)
    
    indices = np.random.choice(len(X_test_chars), 10, replace=False)
    
    for i, ax in enumerate(axes.flat):
        idx = indices[i]
        img = X_test_chars[idx]
        
        pred = char_cnn.predict(img.reshape(1, 28, 28, 1), verbose=0)
        pred_label = np.argmax(pred[0])
        conf = np.max(pred[0]) * 100
        
        ax.imshow(img.squeeze(), cmap='gray')
        color = 'lime' if pred_label == y_test_chars[idx] else 'red'
        ax.set_title(f'Pred: {CHAR_LABELS[pred_label]} ({conf:.1f}%)', color=color)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print('Character CNN or EMNIST data not available')

## 7. Prediction Confidence Distribution

In [ ]:
# Analyze prediction confidence for digit model
if digit_cnn is not None:
    sample_size = min(1000, len(X_test_digits))
    sample_indices = np.random.choice(len(X_test_digits), sample_size, replace=False)
    
    preds = digit_cnn.predict(X_test_digits[sample_indices], verbose=0)
    confidences = np.max(preds, axis=1)
    pred_labels = np.argmax(preds, axis=1)
    true_labels = y_test_digits[sample_indices]
    
    correct = pred_labels == true_labels
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Confidence distribution
    axes[0].hist(confidences[correct], bins=30, alpha=0.7, label='Correct', color='lime')
    axes[0].hist(confidences[~correct], bins=30, alpha=0.7, label='Wrong', color='red')
    axes[0].set_xlabel('Confidence')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Prediction Confidence Distribution')
    axes[0].legend()
    
    # Accuracy by confidence threshold
    thresholds = np.arange(0.5, 1.0, 0.05)
    accuracies = []
    coverages = []
    
    for t in thresholds:
        mask = confidences >= t
        if mask.sum() > 0:
            acc = (pred_labels[mask] == true_labels[mask]).mean()
            cov = mask.mean()
        else:
            acc = 0
            cov = 0
        accuracies.append(acc)
        coverages.append(cov)
    
    axes[1].plot(thresholds, accuracies, 'o-', label='Accuracy', linewidth=2)
    axes[1].plot(thresholds, coverages, 's-', label='Coverage', linewidth=2)
    axes[1].set_xlabel('Confidence Threshold')
    axes[1].set_ylabel('Ratio')
    axes[1].set_title('Accuracy vs Coverage by Confidence Threshold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 8. Hardest Cases (Lowest Confidence Correct Predictions)

In [ ]:
if digit_cnn is not None:
    # Find correct predictions with lowest confidence
    correct_indices = np.where(correct)[0]
    correct_confidences = confidences[correct_indices]
    
    # Sort by confidence (ascending)
    sorted_idx = np.argsort(correct_confidences)[:10]
    hard_indices = sample_indices[correct_indices[sorted_idx]]
    
    fig, axes = plt.subplots(2, 5, figsize=(12, 6))
    fig.suptitle('Hardest Correct Predictions (Lowest Confidence)', fontsize=14)
    
    for i, ax in enumerate(axes.flat):
        idx = hard_indices[i]
        img = X_test_digits[idx]
        
        pred = digit_cnn.predict(img.reshape(1, 28, 28, 1), verbose=0)
        pred_label = np.argmax(pred[0])
        conf = np.max(pred[0]) * 100
        
        ax.imshow(img.squeeze(), cmap='gray')
        ax.set_title(f'{y_test_digits[idx]} → {pred_label} ({conf:.1f}%)')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

## Summary

This notebook demonstrated:
- Loading and comparing trained models
- Running inference on test data
- Analyzing prediction confidence
- Identifying challenging cases